# 01 — Build LoRA Release (Data ETL)

Thin orchestration only. All logic lives in `LoRA/data/`.

raw datasets -> ingest -> normalize -> dedupe -> eval lock -> filter/crop -> caption -> split -> export -> validate

## 1. Install + clone

In [ ]:
!pip install -q 'pandas>=2.0' 'pyarrow>=14' 'imagehash>=4.3.1' 'pillow>=10' pyyaml
import subprocess, sys
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull'], check=True)
sys.path.insert(0, str(REPO))  # repo root -> `import LoRA.*`
print('repo ready')

## 2. Resolve dataset mounts into configs/sources.yaml (if needed)

In [ ]:
import yaml
SRC = REPO/'LoRA'/'configs'/'sources.yaml'
cfg = yaml.safe_load(open(SRC))
# Candidate bases per source (both /kaggle/input/<slug> and /datasets/<user>/<slug> forms)
CAND = {
  'citypersons': ['/kaggle/input/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir',
                  '/kaggle/input/datasets/muttahirulislam/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir',
                  '/kaggle/input/citypersons-canonical'],
  'mot17_02':   ['/kaggle/input/mot17-02-fcrnn','/kaggle/input/datasets/kyoru4444/mot17-02-fcrnn'],
  'human_detection': ['/kaggle/input/human-detection-dataset/human detection dataset',
                      '/kaggle/input/datasets/constantinwerner/human-detection-dataset/human detection dataset'],
}
for s in cfg['sources']:
    for base in CAND.get(s['source_id'], [s['kaggle_mount']]):
        if Path(base).exists():
            s['kaggle_mount'] = base; break
    print(s['source_id'], '->', s['kaggle_mount'], 'OK' if Path(s['kaggle_mount']).exists() else 'MISSING')
SRC_RES = Path('/kaggle/working/sources_resolved.yaml')
yaml.safe_dump(cfg, open(SRC_RES,'w'), sort_keys=False, allow_unicode=True)
print('resolved ->', SRC_RES)

## 3. Run ETL

In [ ]:
from LoRA.data.pipeline import run_full_etl
WORK = Path('/kaggle/working/vin_lora')
result = run_full_etl(WORK, sources_path=SRC_RES,
                      prompts_path=REPO/'LoRA'/'configs'/'prompt_templates.yaml',
                      repo_dir=REPO)
assert result['report']['valid'], result['report']['errors']
RELEASE_DIR = result['release_dir']
print('RELEASE:', RELEASE_DIR)
print('EVAL:   ', result['eval_root'])

## 4. Inspect release.json

In [ ]:
import json
print(json.dumps(json.load(open(RELEASE_DIR/'release.json')), indent=2))